# 13 - When does accDM become indistinguishable from CDM?

The accelerated-DM daughter is born with a velocity kick `v = sqrt(eta(eta+2))/(1+eta)`
set by the energy boost `eta_acc`. Adopting the physical relation **`eta = 1e11 / m`**
(`m` = daughter mass in GeV), a heavier daughter has smaller `eta`, free-streams less,
and is colder. This notebook sweeps `m` upward and finds the mass above which accDM's
matter power `P(k)` and lensed CMB spectra match "CDM" within our thresholds.

Two references ("CDM"): (1) the **cold limit** of the same model (`eta -> 0`), isolating
free-streaming; (2) **plain LCDM** (no acc species). Two metric families: (A) a fixed
fractional tolerance on the spectra, and (B) a cosmic-variance-limited chi^2 detectability.

Daughter on exact quadrature (fluid closure is unusable). Hybrid notebook: inline unit
asserts for the pure metrics (nbmake) + a CLASS scan + diagnostic plots. Run in the
`accDM` classy environment: `pytest --nbmake notebooks_test/13_test_accDM_CDM_indistinguishability.ipynb`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 300})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# ---- Planck 2018 base cosmology ----------------------------------
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}

# ---- fixed accDM decay sector ------------------------------------
KAPPA, A_T = 2.0, 0.13
A_REC = 1.0 / (1.0 + 1090.0)
ETA_COLD = 1e-12                     # "cold limit" eta

# ---- scan axes ---------------------------------------------------
F_SEQ = [0.01, 0.05, 0.1]
MASS_GRID = np.logspace(11, 19, 12)  # GeV  -> eta ~ 1 down to ~1e-8

# ---- observables -------------------------------------------------
K_NODES = np.logspace(-3, 0.0, 60)   # 1/Mpc, sub-horizon at z=0
L_MAX = 2500
Q_BINS = 250                          # daughter momentum bins (accuracy vs speed knob)

PREC_COMMON = {'evolver': 0, 'reionization_z_start_max': 80, 'background_Nloga': 2001}
PREC_PK  = {**PREC_COMMON, 'output': 'mPk', 'P_k_max_1/Mpc': 1.0, 'z_max_pk': 0.0}
PREC_CMB = {**PREC_COMMON, 'output': 'tCl,pCl,lCl,mPk', 'lensing': 'yes',
            'l_max_scalars': L_MAX, 'P_k_max_1/Mpc': 1.0, 'z_max_pk': 0.0}

# ---- survey / detectability assumptions --------------------------
V_SURVEY = 100.0**3                   # (100 Mpc)^3 fiducial; edit to taste
F_SKY = 0.7

def eta_of(m):
    return 1e11 / m


In [ ]:
def ocdm_rescaled(f_acc):
    """CDM density rescaled for the decayed daughter, as in notebook 5."""
    return omega_cdm0 * (1 + f_acc*(1 - A_REC**KAPPA)/(1 + (A_REC/A_T)**KAPPA))**(-1)

def lcdm_params():
    """Plain LCDM, no acc species; a massive-nu sector matched to the accDM runs."""
    p = dict(base_params); p.update(PREC_CMB)
    p.update({'N_ncdm': 1, 'deg_ncdm': 3, 'm_ncdm': 0.02, 'T_ncdm': 0.71611,
              'ncdm_quadrature_strategy': 0, 'ncdm_N_momentum_bins': 15, 'N_ur': 0.00441})
    return p

def _accdm_common(f_acc, m, eta):
    p = dict(base_params); p.update(PREC_CMB)
    p.update({'omega_cdm': ocdm_rescaled(f_acc),
              'vary_Gamma_acc': 'yes', 'kappa_acc': KAPPA, 'a_t_acc': A_T,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': m, 'm_cdm_in_GeV': m,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(m*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, {:d}'.format(Q_BINS), 'N_ur': 0.00441,
              'ncdm_fluid_approximation': 0,
              'ncdm_fluid_trigger_rho_accDM_over_rho_dcdm': 10})
    return p

def accdm_params(f_acc, m):
    return _accdm_common(f_acc, m, eta_of(m))

def coldlimit_params(f_acc):
    return _accdm_common(f_acc, MASS_GRID[-1], ETA_COLD)


In [ ]:
# structural sanity: mapping + required keys present, exact quadrature enforced
assert abs(eta_of(1e11) - 1.0) < 1e-12
assert abs(eta_of(1e18) - 1e-7) < 1e-15
_p = accdm_params(0.1, 1e15)
assert _p['ncdm_quadrature_strategy'].endswith('4')      # daughter exact
assert _p['ncdm_fluid_approximation'] == 0               # never fluid
assert _p['m_ncdm'].split(',')[1].strip() == '{:.6e}'.format(1e15*1e9)
assert _p['eta_acc'] == eta_of(1e15)
assert coldlimit_params(0.1)['eta_acc'] == ETA_COLD
print('Task 1 param builders OK')
